In [8]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import json

# Добавляем корень проекта в пути, чтобы импорты src работали корректно
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.llm_platform.data_foundry.evol_pipeline import EvolPipeline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
input_dataset = project_root / "data/processed/test_sft_dataset.jsonl"
output_dataset = project_root / "data/processed/test_evolved_dataset.jsonl"
prompts_file = project_root / "src/llm_platform/data_foundry/evol_prompts.yaml"

# Инициализируем пайплайн
pipeline = EvolPipeline(
    input_file=input_dataset,
    output_file=output_dataset,
    prompts_file=prompts_file,
    model_name="openai/gpt-oss-20b:free",
    max_concurrent_requests=1 # Можно увеличить до 5, если API стабилен
)

# Запускаем эволюцию
await pipeline.run_evolution()

2026-06-02 18:21:49 [INFO] (src.llm_platform.data_foundry.llm_client): LLMClient initialized with model: openai/gpt-oss-20b:free
2026-06-02 18:21:49 [INFO] (src.llm_platform.data_foundry.evol_pipeline): Loaded 81 base pairs for evolution.
2026-06-02 18:21:50 [INFO] (httpx): HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-06-02 18:21:50 [INFO] (openai._base_client): Retrying request to /chat/completions in 0.387179 seconds
2026-06-02 18:21:51 [INFO] (httpx): HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-06-02 18:21:51 [INFO] (openai._base_client): Retrying request to /chat/completions in 0.999849 seconds
2026-06-02 18:21:52 [INFO] (httpx): HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-02 18:22:18 [INFO] (httpx): HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-02 18:22:32 [INFO] (httpx): HTT

In [12]:
with open(output_dataset, "r", encoding="utf-8") as f:
    first_line = f.readline()
    if first_line:
        evolved_pair = json.loads(first_line)
        
        print(f"ID новой пары: {evolved_pair.get('pair_id')}")
        print(f"Связанный чанк текста: {evolved_pair.get('source_chunk_id')}")
        print(f"Флаг эволюции: {evolved_pair.get('is_evolved')}\n")
        print("-" * 50)
        
        for msg in evolved_pair.get("messages", []):
            role = msg.get("role").upper()
            content = msg.get("content")
            print(f"[{role}]:\n{content}\n")
            print("-" * 50)
    else:
        print("Файл пуст. Проверь логи генерации.")

ID новой пары: evol_ce047d12
Связанный чанк текста: chunk_610154
Флаг эволюции: True

--------------------------------------------------
[ASSISTANT]:
**Новый вопрос:**

*Каковы архитектурные компромиссы, которые приводят к тому, что при средних размерах пакета Mixture of Experts (MoE) модели демонстрируют более выраженную эффективность speculative decoding (SD) по сравнению с плотными моделями, и какие механизмы внутри MoE обеспечивают это преимущество?*

**Ответ:**

При средних размерах пакета (≈ 32–128 токенов) speculative decoding (SD) в Mixture of Experts (MoE) моделях даёт более заметный прирост производительности, чем в плотных моделях, по нескольким взаимосвязанным архитектурным причинам:

1. **Разделённая вычислительная нагрузка** – В MoE каждый токен обрабатывается только одним из нескольких экспертов. При SD модель сначала генерирует «предположительные» токены на более быстрой, но менее точной «первичной» модели, а затем проверяет их на «проверочной» модели. Поскольку каждый 